# AI Regulation and Data Center Siting — Local Jupyter Notebook

**Designed for regular Jupyter Notebook or JupyterLab** with a local clone of the `khalid-add-datasets` GitHub branch. No Colab or Google Drive dependencies. The original Stage 1 cells are retained as optional, disabled cells; Stage 2 reads prepared files from the repository.

In a terminal:

```bash
git clone --branch khalid-add-datasets https://github.com/chalseokorom/ai-law-datacenter-siting.git
cd ai-law-datacenter-siting
python -m pip install pandas numpy geopandas rasterio rasterstats duckdb requests pyarrow scikit-learn jupyter
jupyter lab
```

If the repository is already cloned, run `git checkout khalid-add-datasets` and `git pull`. Open this notebook inside the repository. If Jupyter starts elsewhere, set `REPO_DIR_OVERRIDE` in the Stage 2 setup cell. **Run All** processes the prepared Stage 2 data without rerunning the large CERF rasters. Data files must actually be present locally; Git LFS users may need `git lfs pull`.

The Atlas is **existing** data center records; projected IM3 scenarios remain a separate dataset. Atlas county assignment requires verified geographic columns.

In [ ]:
# Local Jupyter setup. Stage 1 raster extraction is optional and disabled by default.
RUN_STAGE1 = False
print("Stage 1 enabled:", RUN_STAGE1)

In [ ]:
# Repository path is initialized in Stage 2 below.
print("Stage 1 is optional; continue to Stage 2 to use prepared datasets.")

# Stage 1: Optional CERF raster processing

Original extraction and DuckDB cells are preserved but skipped by default. Set `RUN_STAGE1 = True` in the first cell only if you want to rerun the expensive raster workflow. Stage 2 uses the prepared county CSV instead.

In [ ]:
# Optional Stage 1 cell (original command preserved):
# !pip -q install geopandas rasterio rasterstats

In [ ]:
if RUN_STAGE1:
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import rasterio
    from rasterstats import zonal_stats

In [ ]:
if RUN_STAGE1:
    pass  # Original cell contained only comments.
    # # Project paths
    # PROJECT_DIR = Path("/content/CERF_Data_Centers_Project_v1")
    # DATA_DIR = PROJECT_DIR / "cerf_data_centers_package_data"
    # OUTPUT_DIR = PROJECT_DIR / "processed"
    # OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # # Find CERF rasters
    # tif_files = sorted(DATA_DIR.rglob("*.tif"))

    # print(f"Data folder exists: {DATA_DIR.exists()}")
    # print(f"Found {len(tif_files)} TIFF files")
    # for file in tif_files:
    #     print(file.name)

In [ ]:
if RUN_STAGE1:
    # Profile rasters safely by block
    def profile_raster(raster_path):
        valid_count = 0
        nodata_count = 0
        min_value = None
        max_value = None
        value_sum = 0.0

        with rasterio.open(raster_path) as src:
            for _, window in src.block_windows(1):
                data = src.read(1, window=window, masked=True)
                valid = data.compressed()

                valid_count += valid.size
                nodata_count += data.size - valid.size

                if valid.size == 0:
                    continue

                block_min = float(valid.min())
                block_max = float(valid.max())
                min_value = block_min if min_value is None else min(min_value, block_min)
                max_value = block_max if max_value is None else max(max_value, block_max)
                value_sum += valid.astype("float64").sum()

            mean_value = value_sum / valid_count if valid_count else np.nan

            return {
                "file_name": raster_path.name,
                "file_size_mb": round(raster_path.stat().st_size / (1024**2), 2),
                "width": src.width,
                "height": src.height,
                "bands": src.count,
                "dtype": src.dtypes[0],
                "nodata": src.nodata,
                "crs": str(src.crs),
                "resolution_x": src.res[0],
                "resolution_y": src.res[1],
                "valid_cell_count": valid_count,
                "nodata_cell_count": nodata_count,
                "min_value": min_value,
                "max_value": max_value,
                "mean_value": mean_value,
            }

    raster_profile = pd.DataFrame([profile_raster(file) for file in tif_files])
    raster_profile


In [ ]:
if RUN_STAGE1:
    # Load U.S. county boundaries
    COUNTY_URL = "https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip"
    counties = gpd.read_file(COUNTY_URL)[
        ["GEOID", "STATEFP", "COUNTYFP", "NAME", "NAMELSAD", "geometry"]
    ].copy()

    counties = counties.rename(columns={"GEOID": "county_fips", "NAME": "county_name"})
    counties["STATEFP"] = counties["STATEFP"].astype(str).str.zfill(2)
    counties["COUNTYFP"] = counties["COUNTYFP"].astype(str).str.zfill(3)
    counties["county_fips"] = counties["county_fips"].astype(str).str.zfill(5)

    print(f"Loaded {len(counties):,} counties")
    print("County CRS:", counties.crs)
    counties.head()

In [ ]:
if RUN_STAGE1:
    # County aggregation rules
    raster_config = {
        "commercial_electricity_rates_dollar_per_kwh.tif": {"type": "continuous", "stats": ["mean", "median"]},
        "industrial_electricity_rates_dollar_per_kwh.tif": {"type": "continuous", "stats": ["mean", "median"]},
        "distance_to_substation.tif": {"type": "continuous", "stats": ["mean", "min", "median"]},
        "land_value_dollar_per_sqft.tif": {"type": "continuous", "stats": ["mean", "median"]},
        "sales_tax_rate.tif": {"type": "continuous", "stats": ["mean", "median"]},
        "real_property_tax_rate.tif": {"type": "continuous", "stats": ["mean", "median"]},
        "personal_property_tax_rate.tif": {"type": "continuous", "stats": ["mean", "median"]},
        "water_cooling_fraction.tif": {"type": "continuous", "stats": ["mean", "median"]},
        "mechanical_cooling_fraction.tif": {"type": "continuous", "stats": ["mean", "median"]},
        "composite_siting_suitability.tif": {"type": "continuous", "stats": ["mean", "median", "max"]},
        "state_siting_region.tif": {"type": "categorical", "stats": ["majority"]},
        "data_center_market_locations.tif": {"type": "continuous", "stats": ["max,","sum"]},
    }

    print(f"Configured {len(raster_config)} rasters")


In [ ]:
if RUN_STAGE1:
    # Add county centroids
    counties_geo = counties.to_crs("EPSG:4326").copy()
    counties_projected = counties.to_crs("EPSG:5070")
    centroids = counties_projected.geometry.centroid
    centroids_geo = gpd.GeoSeries(centroids, crs="EPSG:5070").to_crs("EPSG:4326")

    counties_geo["centroid_longitude"] = centroids_geo.x
    counties_geo["centroid_latitude"] = centroids_geo.y

    counties_geo[[
        "county_fips",
        "county_name",
        "centroid_longitude",
        "centroid_latitude",
    ]].head()


In [ ]:
if RUN_STAGE1:
    pass  # Original cell contained only comments.
    # # Start with the county table
    # cerf_county = counties_geo.copy()

    # # Process each raster
    # for raster_path in tif_files:

    #     file_name = raster_path.name

    #     # Skip rasters not defined in config
    #     if file_name not in raster_config:
    #         print(f"Skipping: {file_name}")
    #         continue

    #     print(f"Processing: {file_name}")

    #     config = raster_config[file_name]

    #     with rasterio.open(raster_path) as src:

    #         # Match counties to raster CRS
    #         counties_raster = counties_geo.to_crs(src.crs)

    #         # Continuous rasters
    #         if config["type"] == "continuous":

    #             stats = zonal_stats(
    #                 counties_raster.geometry,
    #                 raster_path,
    #                 stats=config["stats"],
    #                 nodata=src.nodata,
    #                 all_touched=False
    #             )

    #             base_name = raster_path.stem

    #             for stat in config["stats"]:

    #                 column_name = f"{base_name}_{stat}"

    #                 cerf_county[column_name] = [
    #                     row.get(stat, np.nan)
    #                     for row in stats
    #                 ]

    #         # Categorical rasters
    #         elif config["type"] == "categorical":

    #             stats = zonal_stats(
    #                 counties_raster.geometry,
    #                 raster_path,
    #                 stats=["majority"],
    #                 nodata=src.nodata,
    #                 all_touched=False
    #             )

    #             base_name = raster_path.stem

    #             column_name = f"{base_name}_majority"

    #             cerf_county[column_name] = [
    #                 row.get("majority", np.nan)
    #                 for row in stats
    #             ]

    # print("Finished extracting raster values.")

In [ ]:
if RUN_STAGE1:
    print("Rows:", cerf_county.shape[0])
    print("Columns:", cerf_county.shape[1])

    cerf_county.head()

In [ ]:
if RUN_STAGE1:
    cerf_county.columns.tolist()

In [ ]:
if RUN_STAGE1:
    cerf_tabular = cerf_county.drop(
        columns=["geometry"]
    ).copy()

    cerf_tabular.head()

In [ ]:
if RUN_STAGE1:
    pass  # Original cell contained only comments.
    # from pathlib import Path

    # output_dir = Path(
    #     "/content/CERF_Data_Centers_Project_v1/processed"
    # )

    # output_dir.mkdir(
    #     parents=True,
    #     exist_ok=True
    # )

    # cerf_tabular.to_csv(
    #     output_dir / "cerf_county_tabular.csv",
    #     index=False
    # )

    # cerf_county.to_file(
    #     output_dir / "cerf_county_mapping.geojson",
    #     driver="GeoJSON"
    # )

    # print("Saved both files.")

# CERF County-Level Dataset Column Key

## Geographic and Identifier Columns

- **county_fips**  
  Five-digit county FIPS code. This is the unique geographic identifier for each county and is useful for joining to other county-level datasets.

- **STATEFP**  
  Two-digit state FIPS code.

- **COUNTYFP**  
  Three-digit county FIPS code within the state. Combined with `STATEFP`, it forms `county_fips`.

- **county_name**  
  Short county name, such as `Cuming` or `Lancaster`.

- **NAMELSAD**  
  Full county name, such as `Cuming County`.

- **geometry**  
  Full polygon boundary of the county. Used for GIS operations and county-level mapping.

- **centroid_longitude**  
  Longitude of the approximate geographic center of the county. Useful for map points and labels.

- **centroid_latitude**  
  Latitude of the approximate geographic center of the county. Useful for map points and labels.

---

## Commercial Electricity Rate

- **commercial_electricity_rates_dollar_per_kwh_mean**  
  Average commercial electricity price, in dollars per kWh, across raster cells within the county.

- **commercial_electricity_rates_dollar_per_kwh_median**  
  Middle commercial electricity rate within the county. Less affected by unusually high or low values than the mean.

---

## Industrial Electricity Rate

- **industrial_electricity_rates_dollar_per_kwh_mean**  
  Average industrial electricity price, in dollars per kWh, across the county.

- **industrial_electricity_rates_dollar_per_kwh_median**  
  Middle industrial electricity rate within the county.

---

## Distance to Substation

- **distance_to_substation_mean**  
  Average distance to a substation across the county.

- **distance_to_substation_min**  
  Smallest distance to a substation found within the county. Represents the closest raster location to substation infrastructure.

- **distance_to_substation_median**  
  Middle distance-to-substation value across the county.

---

## Land Value

- **land_value_dollar_per_sqft_mean**  
  Average land value per square foot across the county.

- **land_value_dollar_per_sqft_median**  
  Middle land value per square foot within the county.

---

## Sales Tax Rate

- **sales_tax_rate_mean**  
  Average sales-tax rate represented across the county.

- **sales_tax_rate_median**  
  Middle sales-tax rate represented within the county.

---

## Real Property Tax Rate

- **real_property_tax_rate_mean**  
  Average real-property tax rate across the county.

- **real_property_tax_rate_median**  
  Middle real-property tax rate within the county.

---

## Personal Property Tax Rate

- **personal_property_tax_rate_mean**  
  Average personal-property tax rate represented across the county.

- **personal_property_tax_rate_median**  
  Middle personal-property tax rate within the county.

---

## Water Cooling Fraction

- **water_cooling_fraction_mean**  
  Average water-cooling fraction across the county.

- **water_cooling_fraction_median**  
  Middle water-cooling fraction across the county.

---

## Mechanical Cooling Fraction

- **mechanical_cooling_fraction_mean**  
  Average mechanical-cooling fraction across the county.

- **mechanical_cooling_fraction_median**  
  Middle mechanical-cooling fraction across the county.

---

## Composite Siting Suitability

- **composite_siting_suitability_mean**  
  Average CERF composite siting-suitability value across the county.

- **composite_siting_suitability_median**  
  Middle siting-suitability value within the county.

- **composite_siting_suitability_max**  
  Highest siting-suitability value found anywhere within the county. Useful for identifying counties that contain at least one highly suitable area.

---

## State Siting Region

- **state_siting_region_majority**  
  Most common state siting-region category found within the county.

  This raster is categorical, so the majority category is used instead of calculating an average.

---

## Data Center Market Locations

- **data_center_market_locations_majority**  
  Most common data-center-market-location category found within the county.

  This raster is categorical, so the majority category is used instead of calculating an average.

---

## Statistic Key

- **mean**  
  Average of all valid raster-cell values inside the county.

- **median**  
  Middle value of the raster cells inside the county. Less sensitive to extreme values than the mean.

- **min**  
  Smallest raster value found inside the county.

- **max**  
  Largest raster value found inside the county.

- **majority**  
  Most frequently occurring category inside the county.

---

## Modeling Note

The geographic and identifier fields such as `county_fips`, county names, latitude, longitude, and `geometry` are mainly used for identification, joining datasets, and mapping.

The electricity, distance, land value, tax, cooling, market

In [ ]:
if RUN_STAGE1:
    # =========================================================
    # NON-AGGREGATED RASTER-LEVEL DATASET
    # =========================================================

    from pathlib import Path
    import rasterio
    import numpy as np
    import pandas as pd
    from rasterio.transform import xy


    # -----------------------------
    # Paths
    # -----------------------------

    PROJECT_DIR = Path("/content/drive/MyDrive/CERF_Data_Centers_Project_v1")

    DATA_DIR = (
        PROJECT_DIR /
        "cerf_data_centers_package_data"
    )

    OUTPUT_DIR = PROJECT_DIR / "processed"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


    # -----------------------------
    # Find raster files
    # -----------------------------

    tif_files_raw = list(DATA_DIR.rglob("*.tif"))

    print(f"Found {len(tif_files_raw)} TIFF files")

    for file in tif_files_raw:
        print(file.name)

In [ ]:
if RUN_STAGE1:
    # -----------------------------
    # Check raster alignment
    # -----------------------------

    reference_path = tif_files_raw[0]

    with rasterio.open(reference_path) as ref:
        reference_crs = ref.crs
        reference_transform = ref.transform
        reference_width = ref.width
        reference_height = ref.height

    same_grid = True

    for raster_path in tif_files_raw:

        with rasterio.open(raster_path) as src:

            matches = (
                src.crs == reference_crs
                and src.transform == reference_transform
                and src.width == reference_width
                and src.height == reference_height
            )

            print(
                raster_path.name,
                "MATCH" if matches else "DIFFERENT GRID"
            )

            if not matches:
                same_grid = False

    print()
    print("All rasters aligned:", same_grid)

In [ ]:
if RUN_STAGE1:
    pass  # Original cell contained only comments.
    # # -----------------------------
    # # Extract raw raster-cell values
    # # -----------------------------

    # sources = {
    #     path.stem: rasterio.open(path)
    #     for path in tif_files_raw
    # }

    # reference = sources[reference_path.stem]

    # part_number = 0
    # saved_files = []

    # for _, window in reference.block_windows(1):

    #     rows, cols = np.indices(
    #         (
    #             int(window.height),
    #             int(window.width)
    #         )
    #     )

    #     rows = rows + int(window.row_off)
    #     cols = cols + int(window.col_off)

    #     xs, ys = xy(
    #         reference.transform,
    #         rows,
    #         cols,
    #         offset="center"
    #     )

    #     block = pd.DataFrame({
    #         "raster_row": rows.ravel(),
    #         "raster_col": cols.ravel(),
    #         "longitude": np.asarray(xs).ravel(),
    #         "latitude": np.asarray(ys).ravel()
    #     })

    #     valid_any = np.zeros(
    #         len(block),
    #         dtype=bool
    #     )

    #     for name, src in sources.items():

    #         data = src.read(
    #             1,
    #             window=window,
    #             masked=True
    #         )

    #         values = (
    #             data
    #             .astype("float64")
    #             .filled(np.nan)
    #             .ravel()
    #         )

    #         block[name] = values

    #         valid_any = valid_any | ~np.isnan(values)

    #     # Remove locations with no raster data
    #     block = block.loc[valid_any].copy()

    #     if block.empty:
    #         continue

    #     part_path = (
    #         OUTPUT_DIR /
    #         f"cerf_raw_cells_part_{part_number:05d}.parquet"
    #     )

    #     block.to_parquet(
    #         part_path,
    #         index=False
    #     )

    #     saved_files.append(part_path)

    #     part_number += 1

    #     if part_number % 100 == 0:
    #         print(f"Saved {part_number} blocks")


    # for src in sources.values():
    #     src.close()

    # print()
    # print(f"Finished. Created {len(saved_files)} parquet files.")

In [ ]:
if RUN_STAGE1:
    pass  # Original cell contained only comments.
    # from pathlib import Path
    # import shutil

    # PROJECT_DIR = Path("/content/CERF_Data_Centers_Project_v1")
    # OUTPUT_DIR = PROJECT_DIR / "processed"

    # RAW_CELLS_DIR = OUTPUT_DIR / "cerf_non_aggregated_tabular"
    # RAW_CELLS_DIR.mkdir(parents=True, exist_ok=True)

    # moved = 0

    # for file in OUTPUT_DIR.glob("cerf_raw_cells_part_*.parquet"):
    #     destination = RAW_CELLS_DIR / file.name

    #     shutil.move(
    #         str(file),
    #         str(destination)
    #     )

    #     moved += 1

    # print(f"Moved {moved} parquet files.")
    # print("New folder:", RAW_CELLS_DIR)

In [ ]:
# Optional Stage 1 cell (original command preserved):
# # =========================================================
# # DUCKDB STORAGE FOR NON-AGGREGATED CERF DATA
# # =========================================================
# 
# # Install DuckDB
# !pip install duckdb -q

In [ ]:
if RUN_STAGE1:
    # =========================================================
    # 1. MOUNT GOOGLE DRIVE
    # =========================================================

    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
if RUN_STAGE1:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
if RUN_STAGE1:
    # =========================================================
    # 2. DEFINE STORAGE PATHS
    # =========================================================

    from pathlib import Path
    import duckdb

    DRIVE_PROJECT_DIR = Path(
        "/content/drive/MyDrive/CERF_Data_Centers_Project_v1"
    )

    DRIVE_RAW_DIR = Path(
        "/content/drive/MyDrive/"
        "CERF_Data_Centers_Project/"
        "cerf_processed_tabular_data/"
        "processed/"
        "cerf_non_aggregated_tabular"
    )

    DRIVE_RAW_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    DUCKDB_PATH = (
        DRIVE_PROJECT_DIR
        / "cerf_non_aggregated.duckdb"
    )

    print("DuckDB path:")
    print(DUCKDB_PATH)

    print("\nParquet folder:")
    print(DRIVE_RAW_DIR)

In [ ]:
if RUN_STAGE1:
    pass  # Original cell contained only comments.
    # # =========================================================
    # # 3. COPY PARQUET FILES FROM COLAB TO GOOGLE DRIVE
    # # =========================================================

    # import shutil

    # LOCAL_RAW_DIR = Path(
    #     "/content/CERF_Data_Centers_Project_v1/processed/"
    #     "cerf_non_aggregated_tabular"
    # )

    # copied = 0

    # for file in LOCAL_RAW_DIR.glob(
    #     "cerf_raw_cells_part_*.parquet"
    # ):

    #     destination = DRIVE_RAW_DIR / file.name

    #     if not destination.exists():

    #         shutil.copy2(
    #             file,
    #             destination
    #         )

    #         copied += 1

    # print(f"Copied {copied} files to Google Drive.")

In [ ]:
if RUN_STAGE1:
    # =========================================================
    # 4. CREATE DUCKDB DATABASE
    # =========================================================

    con = duckdb.connect(
        str(DUCKDB_PATH)
    )

    print("Connected to DuckDB.")

In [ ]:
if RUN_STAGE1:
    pass  # Original cell contained only comments.
    # # =========================================================
    # # 5. CREATE VIEW OF ALL PARQUET FILES
    # # =========================================================

    # parquet_pattern = str(
    #     DRIVE_RAW_DIR / "*.parquet"
    # )

    # print("Reading Parquet files from:")
    # print(parquet_pattern)

    # con.execute(f"""
    # CREATE OR REPLACE VIEW cerf_raw_cells AS
    # SELECT *
    # FROM read_parquet(
    #     '{parquet_pattern}'
    # )
    # """)

    # print("DuckDB view created.")

In [ ]:
if RUN_STAGE1:
    pass  # Original cell contained only comments.
    # =========================================================
    # 6. CHECK TOTAL ROW COUNT
    # =========================================================

    # row_count = con.execute("""
    # SELECT COUNT(*)
    # FROM cerf_raw_cells
    # """).fetchone()[0]

    # print(f"Total rows: {row_count:,}")

In [ ]:
if RUN_STAGE1:
    pass  # Original cell contained only comments.
    # # =========================================================
    # # 7. PREVIEW DATA
    # # =========================================================

    # preview = con.execute("""
    # SELECT *
    # FROM cerf_raw_cells
    # LIMIT 20
    # """).df()

    # preview

In [ ]:
if RUN_STAGE1:
    # =========================================================
    # 10. CLOSE DATABASE CONNECTION
    # =========================================================

    con.close()

    print("DuckDB connection closed.")

## Cleaning and Joining the Data ##

## Stage 2: Clean and join repository datasets (local Jupyter)

Run from a clone of the `khalid-add-datasets` branch. All files are read from the local repository, and processed outputs are saved locally. No Google Drive or Colab is required.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

# The notebook should be opened from somewhere inside the cloned repository.
# If it is outside the repository, set REPO_DIR_OVERRIDE to the full local path.
REPO_DIR_OVERRIDE = None

def find_repo_root(start):
    for folder in [start, *start.parents]:
        if (folder / '.git').exists() or ((folder / 'raw data').is_dir() and (folder / 'README.md').exists()):
            return folder
    raise FileNotFoundError(
        'Repository not found. Open Jupyter from the cloned repository, '
        'or set REPO_DIR_OVERRIDE to its full local path.'
    )

project_dir = (Path(REPO_DIR_OVERRIDE).expanduser().resolve()
               if REPO_DIR_OVERRIDE else find_repo_root(Path.cwd().resolve()))
PROJECT_DIR = project_dir
output_dir = project_dir / 'processed' / 'stage2_clean_join'
output_dir.mkdir(parents=True, exist_ok=True)
print('Repository:', project_dir)
print('Output directory:', output_dir)

Repository: /Users/khalidjackson/Desktop/ai-law-datacenter-siting
Output directory: /Users/khalidjackson/Desktop/ai-law-datacenter-siting/processed/stage2_clean_join


This keeps all the paths together, so you only need to update one place if the Google Drive structure changes.

In [7]:
# Find repository datasets without assuming their subfolders are identical to Google Drive.
# Prefer exact known locations, then search by filename.
def locate_file(filename, preferred=()):
    for relative in preferred:
        candidate = project_dir / relative
        if candidate.is_file():
            return candidate
    matches = sorted(p for p in project_dir.rglob(filename)
                     if p.is_file() and '.git' not in p.parts and 'stage2_clean_join' not in p.parts)
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(f'Missing {filename} under {project_dir}. Check Git LFS and git pull.')
    raise ValueError(f'Multiple files named {filename}: {matches}. Specify a path explicitly.')

cerf_path = locate_file('cerf_county_tabular.csv', ['cerf_processed_tabular_data/cerf_county_tabular.csv'])
fips_path = locate_file('State,_County_and_City_FIPS_Reference_Table_20261005.csv')
ncsl_path = locate_file('ncsl_data_2019_2025_base.csv')
atlas_relative_path = ('raw data/im3_open_source_data_center_atlas_v2026.02.09/'
                       'im3_open_source_data_center_atlas_v2026.02.09.csv')
atlas_path = locate_file('im3_open_source_data_center_atlas_v2026.02.09.csv', [atlas_relative_path])

im3_dirs = [p for p in project_dir.rglob('im3_projected_data_centers_v1.1') if p.is_dir()]
if len(im3_dirs) != 1:
    raise FileNotFoundError(f'Expected one IM3 projected scenario folder; found {len(im3_dirs)}: {im3_dirs}')
im3_dir = im3_dirs[0]

for name, path in {'CERF':cerf_path,'FIPS':fips_path,'NCSL':ncsl_path,
                   'IM3 Projected':im3_dir,'IM3 Atlas':atlas_path}.items():
    print(f'{name}: {path.relative_to(project_dir)}')

FileNotFoundError: Missing cerf_county_tabular.csv under /Users/khalidjackson/Desktop/ai-law-datacenter-siting. Check Git LFS and git pull.

### Local GitHub repository data

The Atlas and other raw datasets are loaded directly from the cloned repository. No downloads to Google Drive are needed. If files are missing, run `git pull` and, if the repository uses Git LFS, `git lfs pull` in a terminal.

In [ ]:
# Confirm that the local Atlas CSV is a real file and not a Git LFS pointer.
print('Atlas exists:', atlas_path.exists())
print('Atlas size (bytes):', atlas_path.stat().st_size)
with atlas_path.open('rb') as f:
    first_bytes = f.read(100)
if first_bytes.startswith(b'version https://git-lfs.github.com/spec/v1'):
    raise RuntimeError('Atlas is a Git LFS pointer. Run `git lfs pull` from the repository root.')
print('Atlas file is available locally.')

In [ ]:
# Locate prepared files in the local repository.
for keyword in ['cerf_county_tabular', 'im3_open_source_data_center_atlas']:
    matches = [p for p in project_dir.rglob('*') if p.is_file() and keyword.lower() in p.name.lower()]
    print(f'{keyword}:', *[str(p.relative_to(project_dir)) for p in matches], sep='\n  ')

# Find all IM3 GeoJSON files


In [ ]:
# Locate IM3 projection files

im3_files = sorted(im3_dir.rglob('*.geojson'))

print(f'IM3 GeoJSON files found: {len(im3_files)}')

for file in im3_files:
    print(file.relative_to(im3_dir))

Convert GeoJSON files to CSV

In [ ]:
import geopandas as gpd
import pandas as pd

# Folder for converted IM3 datasets
im3_output_dir = output_dir / 'im3_csv'
im3_output_dir.mkdir(parents=True, exist_ok=True)

converted_files = []

for file in im3_files:

    # Read GeoJSON
    gdf = gpd.read_file(file)

    # Preserve scenario information
    gdf['growth_scenario'] = file.parent.name
    gdf['market_gravity'] = file.stem.split('_market_gravity')[0].split('_')[-1]

    # Store geometry as text so spatial information is not lost
    gdf['geometry_wkt'] = gdf.geometry.to_wkt()

    # Convert to regular DataFrame
    df = pd.DataFrame(gdf.drop(columns='geometry'))

    # Save as CSV
    output_file = im3_output_dir / f'{file.stem}.csv'
    df.to_csv(output_file, index=False)

    converted_files.append({
        'scenario': file.parent.name,
        'file': output_file.name,
        'rows': len(df),
        'columns': len(df.columns)
    })

    print(f'Converted: {output_file.name}')

Review converted datasets

In [ ]:
# Summary of converted IM3 files

conversion_summary = pd.DataFrame(converted_files)

display(conversion_summary)

print(f'Total CSV files: {len(conversion_summary)}')
print(f'Total records: {conversion_summary["rows"].sum():,}')

conversion_summary.to_csv(
    im3_output_dir / 'conversion_summary.csv',
    index=False
)

Preview and check each CSV

In [ ]:
# Basic data quality checks

quality_checks = []

for file in sorted(im3_output_dir.glob('*.csv')):

    if file.name == 'conversion_summary.csv':
        continue

    df = pd.read_csv(file)

    quality_checks.append({
        'file': file.name,
        'rows': len(df),
        'columns': len(df.columns),
        'duplicates': int(df.duplicated().sum()),
        'missing_values': int(df.isna().sum().sum()),
        'missing_percent': round(
            df.isna().sum().sum() / df.size * 100, 2
        ) if df.size else 0
    })

quality_summary = pd.DataFrame(quality_checks)

display(quality_summary)

quality_summary.to_csv(
    im3_output_dir / 'quality_summary.csv',
    index=False
)

Load the datasets

In [ ]:
# Load datasets

cerf = pd.read_csv(cerf_path, dtype={'county_fips': 'string'})
fips_ref = pd.read_csv(fips_path, dtype='string')
ncsl = pd.read_csv(ncsl_path, dtype='string')
atlas = pd.read_csv(atlas_path, dtype='string')

# Organize datasets
datasets = {
    'CERF': cerf,
    'FIPS': fips_ref,
    'NCSL': ncsl,
    'IM3 Atlas': atlas
}

# Check dataset sizes
for name, df in datasets.items():
    df.columns = df.columns.str.strip()
    print(f'{name}: {df.shape[0]:,} rows, {df.shape[1]} columns')

We'll use a small number of reusable functions rather than repeating the same code for every dataset.

In [ ]:
def clean_fips(series, length):
    """Format geographic identifiers while preserving leading zeros."""
    values = series.astype('string').str.strip()
    values = values.str.replace(r'\.0$', '', regex=True)

    valid = values.str.fullmatch(rf'\d{{1,{length}}}', na=False)

    return values.where(valid).str.zfill(length)


def check_duplicates(df, column, name):
    duplicates = df[df.duplicated(column, keep=False)]

    if not duplicates.empty:
        duplicates.to_csv(
            output_dir / f'{name}_duplicate_keys.csv',
            index=False
        )
        print(f'{name}: {len(duplicates):,} rows with duplicate keys')

    return duplicates


def save_clean(df, filename):
    df.to_csv(output_dir / filename, index=False)
    print(f'Saved {filename}: {len(df):,} rows')

Our FIPS reference includes multiple cities within the same county. We only need one record per county.

In [ ]:
fips_clean = fips_ref[
    ['StCnty FIPS Code', 'State FIPS Code',
     'State Code', 'State Name', 'County Name']
].copy()

fips_clean.columns = [
    'county_fips', 'state_fips',
    'state_abbr', 'state_name', 'county_name'
]

fips_clean['county_fips'] = clean_fips(fips_clean['county_fips'], 5)
fips_clean = fips_clean.dropna(subset=['county_fips'])

fips_clean['state_fips'] = fips_clean['county_fips'].str[:2]

# Check whether repeated county rows disagree on county information
county_conflicts = (
    fips_clean.groupby('county_fips')
    [['state_abbr', 'state_name', 'county_name']]
    .nunique(dropna=True)
)

county_conflicts = county_conflicts[
    county_conflicts.gt(1).any(axis=1)
]

county_conflicts.to_csv(output_dir / 'fips_conflicts.csv')

# Repeated city rows are expected in this reference
fips_clean = fips_clean.drop_duplicates('county_fips')

save_clean(fips_clean, '01_fips_county_clean.csv')

print('Conflicting county identifiers:', len(county_conflicts))
fips_clean.head()

We're using your already aggregated county-level CERF file here. This cell cleans it without aggregating it again or removing its existing statistical features.

In [ ]:
cerf_clean = cerf.copy()

cerf_clean['county_fips'] = clean_fips(
    cerf_clean['county_fips'], 5
)

# Remove rows without a valid county identifier
cerf_invalid = cerf_clean[cerf_clean['county_fips'].isna()]
cerf_invalid.to_csv(output_dir / 'cerf_invalid_fips.csv', index=False)

cerf_clean = cerf_clean.dropna(subset=['county_fips'])

# County names and geographic identifiers come from the FIPS reference
extra_columns = ['STATEFP', 'COUNTYFP', 'county_name', 'NAMELSAD']
cerf_clean = cerf_clean.drop(
    columns=extra_columns,
    errors='ignore'
)

# CERF feature columns should be numeric
for column in cerf_clean.columns:
    if column != 'county_fips':
        cerf_clean[column] = pd.to_numeric(
            cerf_clean[column], errors='coerce'
        )

cerf_clean = cerf_clean.replace([np.inf, -np.inf], np.nan)

cerf_duplicates = check_duplicates(
    cerf_clean, 'county_fips', 'cerf'
)

if not cerf_duplicates.empty:
    raise ValueError('Review duplicate CERF counties before joining.')

save_clean(cerf_clean, '02_cerf_county_clean.csv')

cerf_clean.head()

These two datasets need different transformations: IM3 becomes a county-level count, while NCSL becomes a state-level summary.

In [ ]:
# IM3 Existing Atlas: inspect and aggregate verified county identifiers.
# The Atlas is NOT the projected scenario dataset.
atlas_clean = atlas.copy()
print('Atlas columns:', atlas_clean.columns.tolist())

# Verify the original expected geographic columns before using them.
if 'county_fips' in atlas_clean.columns:
    atlas_clean['county_fips'] = clean_fips(atlas_clean['county_fips'], 5)
elif {'state_id', 'county_id'}.issubset(atlas_clean.columns):
    atlas_clean['state_fips'] = clean_fips(atlas_clean['state_id'], 2)
    atlas_clean['county_code'] = clean_fips(atlas_clean['county_id'], 3)
    atlas_clean['county_fips'] = atlas_clean['state_fips'] + atlas_clean['county_code']
else:
    raise ValueError(
        'Atlas has no verified county_fips or state_id/county_id columns. '
        'Inspect the printed Atlas columns. If it has coordinates, a spatial join '
        'to county boundaries is needed before county aggregation.'
    )

# Audit duplicate facility IDs if an ID column exists.
if 'id' in atlas_clean.columns:
    atlas_clean[atlas_clean.duplicated('id', keep=False)].to_csv(
        output_dir / 'atlas_repeated_ids.csv', index=False
    )
else:
    print('No id column found: facility-level duplicate ID audit skipped.')

# Audit records without a valid reference county.
atlas_unmatched = atlas_clean[
    atlas_clean['county_fips'].isna() |
    ~atlas_clean['county_fips'].isin(fips_clean['county_fips'])
].copy()
atlas_unmatched.to_csv(output_dir / 'atlas_unmatched_counties.csv', index=False)

# Keep only records matched to a reference county for aggregation.
atlas_matched = atlas_clean[
    atlas_clean['county_fips'].isin(fips_clean['county_fips'])
].copy()
atlas_county = (
    atlas_matched.groupby('county_fips')
    .size().reset_index(name='atlas_record_count')
)
atlas_county['has_existing_data_center'] = 1
save_clean(atlas_county, '03_atlas_county_clean.csv')
print('Atlas records:', len(atlas_clean))
print('Matched records:', len(atlas_matched))
print('Unmatched records:', len(atlas_unmatched))
print('Counties with Atlas records:', len(atlas_county))
print('Note: Counts reflect Atlas records, not necessarily verified unique facilities.')


In [ ]:
# NCSL: legislation records to state-level policy features
ncsl_clean = ncsl.copy()

states = fips_clean[
    ['state_name', 'state_fips']
].drop_duplicates('state_fips').copy()

states['state_key'] = states['state_name'].str.upper().str.strip()
ncsl_clean['state_key'] = ncsl_clean['State'].str.upper().str.strip()

ncsl_clean = ncsl_clean.merge(
    states[['state_key', 'state_fips']],
    on='state_key',
    how='left',
    validate='many_to_one'
)

ncsl_clean['Year'] = pd.to_numeric(
    ncsl_clean['Year'], errors='coerce'
)

ncsl_clean['status_clean'] = (
    ncsl_clean['Bill Status']
    .fillna('')
    .str.lower()
    .str.replace('–', '-', regex=False)
    .str.strip()
)

ncsl_clean['is_enacted'] = (
    ncsl_clean['status_clean'].isin(['enacted', 'adopted'])
).astype(int)

ncsl_clean['bill_key'] = (
    ncsl_clean['Bill Number'].fillna('').str.upper().str.strip()
    + '_'
    + ncsl_clean['Year'].astype('Int64').astype('string').fillna('')
)

ncsl_clean = ncsl_clean[
    ncsl_clean['Bill Number'].notna()
].copy()

ncsl_clean[ncsl_clean['state_fips'].isna()].to_csv(
    output_dir / 'ncsl_unmatched_states.csv', index=False
)

# Count each bill once, even if it appears under multiple categories
unique_bills = (
    ncsl_clean.dropna(subset=['state_fips'])
    .drop_duplicates(['state_fips', 'bill_key'])
)

ncsl_state = (
    unique_bills.groupby('state_fips')
    .agg(
        ncsl_bill_count=('bill_key', 'size'),
        ncsl_enacted_count=('is_enacted', 'sum')
    )
    .reset_index()
)

save_clean(ncsl_state, '04_ncsl_state_clean.csv')

ncsl_state.head()

This produces the master dataset that we'll use for EDA and modeling.

In [ ]:
master = fips_clean.merge(
    cerf_clean,
    on='county_fips',
    how='left',
    validate='one_to_one',
    indicator='cerf_join'
)

master = master.merge(
    atlas_county,
    on='county_fips',
    how='left',
    validate='one_to_one',
    indicator='atlas_join'
)

master = master.merge(
    ncsl_state,
    on='state_fips',
    how='left',
    validate='many_to_one',
    indicator='ncsl_join'
)

# A missing Atlas record is not automatically a verified negative label
master['atlas_record_count'] = (
    master['atlas_record_count'].fillna(0).astype(int)
)

# Check join coverage before modeling
for source in ['cerf', 'atlas', 'ncsl']:
    column = f'{source}_join'
    matched = master[column].eq('both').sum()

    print(f'{source.upper()}: {matched:,} / {len(master):,} matched')

print(f'\nMaster dataset: {master.shape[0]:,} rows')
print(f'Total columns: {master.shape[1]}')

if master['county_fips'].duplicated().any():
    raise ValueError('The master dataset contains duplicate counties.')

save_clean(master, 'master_county_clean.csv')

master.head()

This displays the first five records from each dataset, along with the number of rows and columns.

In [ ]:
# Preview cleaned datasets

cleaned_datasets = {
    'FIPS': fips_clean,
    'CERF': cerf_clean,
    'IM3 Atlas': atlas_county,
    'NCSL': ncsl_state,
    'Master': master
}

for name, df in cleaned_datasets.items():
    print(f'\n{name.upper()} DATASET')
    print(f'Rows: {len(df):,} | Columns: {len(df.columns)}')

    display(df.head(5))

This checks record counts, duplicates, missing values, and column types.

In [ ]:
# High-level data quality checks

quality_results = []

for name, df in cleaned_datasets.items():

    key = 'state_fips' if name == 'NCSL' else 'county_fips'

    quality_results.append({
        'Dataset': name,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Duplicate Rows': df.duplicated().sum(),
        'Duplicate Keys': df[key].duplicated().sum(),
        'Missing Values': df.isna().sum().sum(),
        'Missing %': round(
            df.isna().sum().sum() / df.size * 100, 2
        ) if df.size else 0,
        'Numeric Columns': len(df.select_dtypes(include='number').columns),
        'Text Columns': len(df.select_dtypes(include=['object', 'string']).columns)
    })

quality_summary = pd.DataFrame(quality_results)

display(quality_summary)


# Columns with the most missing values

for name, df in cleaned_datasets.items():

    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print(f'\n{name.upper()} — MISSING VALUES')

    if missing.empty:
        print('No missing values found.')
    else:
        display(
            pd.DataFrame({
                'Missing Count': missing,
                'Missing %': (missing / len(df) * 100).round(2)
            }).head(15)
        )


# Master dataset join coverage

print('\nMASTER DATASET — JOIN COVERAGE')

for source in ['cerf', 'atlas', 'ncsl']:
    column = f'{source}_join'

    print(f'\n{source.upper()}')
    display(master[column].value_counts(dropna=False))

## Local GitHub file inspection

These cells replace the old GitHub API download steps. Since you are running a notebook inside a cloned repository, you can inspect files and use them directly.

In [ ]:
# List raw data files already available locally.
raw_data_dir = project_dir / 'raw data'
if not raw_data_dir.is_dir():
    raise FileNotFoundError(f'Expected raw data folder: {raw_data_dir}')
raw_files = sorted(p for p in raw_data_dir.rglob('*') if p.is_file())
print(f'Found {len(raw_files)} local files in raw data/')
for path in raw_files:
    print(path.relative_to(project_dir))

In [ ]:
# Open any CSV in raw data by its repository relative path.
selected_path = ('raw data/im3_open_source_data_center_atlas_v2026.02.09/'
                 'im3_open_source_data_center_atlas_v2026.02.09.csv')
selected_file = project_dir / selected_path
if not selected_file.is_file():
    raise FileNotFoundError(selected_file)
selected_preview = pd.read_csv(selected_file, nrows=5)
print('Selected:', selected_file.relative_to(project_dir))
print('Columns:', selected_preview.columns.tolist())
display(selected_preview)

In [ ]:
# Quick check of Atlas source and its county identifiers after Stage 2 runs.
print('Atlas path:', atlas_path)
print('Atlas loaded:', 'atlas' in globals())
if 'atlas' in globals():
    print('Atlas shape:', atlas.shape)
    display(atlas.head(3))
if 'atlas_county' in globals():
    print('County-level Atlas shape:', atlas_county.shape)
    print('Duplicate county keys:', atlas_county['county_fips'].duplicated().sum())
    display(atlas_county.head(3))
if 'master' in globals():
    print('Master counties:', len(master))
    print('Atlas matched counties:', master['atlas_join'].eq('both').sum())
    print('CERF matched counties:', master['cerf_join'].eq('both').sum())
